<a id='map-filter'></a>

## 11. 🧩 Pattern 11: map() and filter() — Lazy Transformers — LC 1, 15, 49, 125, 242, 344, 383

---

```
PROBLEM:
  LC 1   — Two Sum: parse input line "2 7 11 15" → list of ints
  LC 15  — 3Sum: input parsing via map(int, input().split())
  LC 49  — Group Anagrams: map chars to sorted signature
  LC 125 — Valid Palindrome: filter(str.isalnum, s) — drop non-alphanumeric
  LC 242 — Valid Anagram: map(str.lower, s) — normalize case
  LC 344 — Reverse String: map over char array
  LC 383 — Ransom Note: filter chars by availability

SYNTAX:
  map(fn, iterable)      → lazy iterator of fn(x) for each x
  filter(fn, iterable)   → lazy iterator of x where fn(x) is truthy
  Must wrap in list() to materialize — they are iterators, not lists.

map() vs LIST COMP — both valid, pick by readability:
  map(int, tokens)               ↔  [int(x) for x in tokens]
  map(str.lower, chars)          ↔  [c.lower() for c in chars]
  map(lambda x: x*2, nums)       ↔  [x*2 for x in nums]
  Rule: use map() when fn already exists (int, str.lower, abs).
        use list comp when fn needs to be written inline.

filter() vs LIST COMP — same rule:
  filter(str.isalnum, s)         ↔  [c for c in s if c.isalnum()]
  filter(lambda x: x>0, nums)    ↔  [x for x in nums if x > 0]
  Rule: use filter() when predicate already exists (str.isalnum, str.isdigit).
        use list comp when predicate is a lambda.

LAZY — iterators, not lists:
  m = map(int, ["1","2","3"])
  type(m)           → <class 'map'>    — not a list yet
  list(m)           → [1, 2, 3]        — materialized
  list(m)           → []               — exhausted! iterator is one-use.
  ⚠  Consume a map/filter iterator only ONCE.

CHAINING map + filter:
  # filter then transform — evens doubled
  result = list(map(lambda x: x*2, filter(lambda x: x%2==0, nums)))
  # equivalent list comp (usually cleaner):
  result = [x*2 for x in nums if x%2 == 0]

COMMON INTERVIEW PATTERN — parse space-separated ints:
  line = "2 7 11 15"
  nums = list(map(int, line.split()))   # split → ['2','7','11','15'] → [2,7,11,15]
  # slow motion:
  # line.split()         → ['2', '7', '11', '15']
  # map(int, [...])      → map object (lazy)
  # list(...)            → [2, 7, 11, 15]   ← materialized

SLOW MOTION TRACE — filter(str.isalnum, "a1 b!c"):
  s = "a1 b!c"
  step 1: 'a' → isalnum=True  → keep
  step 2: '1' → isalnum=True  → keep
  step 3: ' ' → isalnum=False → drop
  step 4: 'b' → isalnum=True  → keep
  step 5: '!' → isalnum=False → drop
  step 6: 'c' → isalnum=True  → keep
  result: ['a','1','b','c']   — iterator, not list yet

KEY INSIGHT:
  map() and filter() are lazy — zero work happens until you consume them.
  The killer use case: list(map(int, input().split())) for parsing.
  For everything else, list comp is usually cleaner.

TIME / SPACE:
  Time:  O(n) — one pass, same as list comp
  Space: O(1) — iterator holds no elements; O(n) only after list() call
```

In [ ]:
# Pattern 11: map() and filter()
# Lazy transformers — zero work until consumed. Wrap in list() to materialize.

# 1. map() — transform every element
tokens  = ["1", "2", "3", "4"]
as_ints = list(map(int, tokens))            # built-in fn — cleaner than list comp
doubled = list(map(lambda x: x * 2, as_ints))
lowered = list(map(str.lower, ["Hello", "WORLD", "Foo"]))  # method reference
print(f"str→int : {as_ints}")
print(f"doubled : {doubled}")
print(f"lowered : {lowered}")

# 2. filter() — keep elements where predicate is True
nums    = [-3, 0, 1, -1, 5, 2, -2, 4]
pos     = list(filter(lambda x: x > 0, nums))    # lambda predicate
alnum   = list(filter(str.isalnum, "a1 b!c2"))   # method reference predicate
print(f"positives : {pos}")
print(f"alnum only: {alnum}")

# 3. lazy — iterator is ONE-USE, type is not list
m = map(int, ["10", "20", "30"])
print(f"type before list(): {type(m)}")
print(f"first list()  : {list(m)}")    # [10, 20, 30]
print(f"second list() : {list(m)}")    # [] — exhausted!

# 4. chaining map + filter vs list comp — side-by-side
data = range(1, 11)
via_chain = list(map(lambda x: x * x, filter(lambda x: x % 2 == 0, data)))
via_comp  = [x * x for x in data if x % 2 == 0]   # usually cleaner
print(f"chain  : {via_chain}")
print(f"comp   : {via_comp}")

# 5. parse drill — space-separated ints (the #1 interview input pattern)
line  = "2 7 11 15"
nums2 = list(map(int, line.split()))
# slow motion:
# line.split()        → ['2', '7', '11', '15']
# map(int, [...])     → lazy map object
# list(...)           → [2, 7, 11, 15]
print(f"parsed : {nums2}")

# multi-line input → list of lists
raw   = ["1 2 3", "4 5 6", "7 8 9"]
grid  = [list(map(int, row.split())) for row in raw]
print(f"grid   : {grid}")


def is_palindrome(s: str) -> bool:
    """
    LC 125 — Valid Palindrome
    Approach: filter non-alnum with filter(), lowercase with map(), compare to reverse.
    Args:
        s (str): input string.
    Returns:
        bool: True if s is a palindrome ignoring case and non-alphanumeric chars.
    Time:  O(n) — filter + map + reverse, all O(n)
    Space: O(n) — cleaned string
    """
    cleaned = list(map(str.lower, filter(str.isalnum, s)))

    # slow motion on s = "A man, a plan, a canal: Panama":
    # filter(str.isalnum, s) → 'A','m','a','n','a','p','l','a','n','a','c','a','n','a','l','P','a','n','a','m','a'
    # map(str.lower, ...)    → 'a','m','a','n','a','p','l','a','n','a','c','a','n','a','l','p','a','n','a','m','a'
    # cleaned == cleaned[::-1] → True

    return cleaned == cleaned[::-1]


def test_harness(fn):
    tests = [
        ("A man, a plan, a canal: Panama", True),   # classic palindrome
        ("race a car",                     False),  # not palindrome
        (" ",                              True),   # empty after filter → palindrome
        ("0P",                             False),  # "0p" ≠ "p0"
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(is_palindrome)

print("map_and_filter defined.")